In [34]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from lightgbm import LGBMRegressor
import matplotlib as plt
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [54]:
df = pd.read_csv(r"C:\Users\admin\OneDrive\Desktop\healthcare_ppg\data\processed\features_rr.csv")

y = np.load(r"C:\Users\admin\OneDrive\Desktop\healthcare_ppg\data\processed\y_labels.npy")

print("Features:", df.shape)
print("Labels:", y.shape)

assert len(features_df) == len(y)



Features: (1638, 13)
Labels: (1638,)


In [55]:
X_windows = np.load(r"C:\Users\admin\OneDrive\Desktop\healthcare_ppg\data\processed\X_windows.npy")

print("Shape:", X_windows.shape)


Shape: (1638, 3750)


In [56]:
print("Total windows:", len(y))
print("Total subjects:", 53)
print("Windows per subject (approx):", 1638 / 53)

Total windows: 1638
Total subjects: 53
Windows per subject (approx): 30.90566037735849


In [57]:
df = pd.read_csv(r"C:\Users\admin\OneDrive\Desktop\healthcare_ppg\data\processed\features_rr.csv")

print("Loaded shape:", features_df.shape)
df.head()

Loaded shape: (1638, 13)


,mean,std,skew,kurtosis,rms,ptp,dominant_freq,resp_power,total_power,rr_dominant_freq,rr_resp_power,rr_total_power,RR
0,-0.000089,0.005012,0.232912,-0.641657,0.005013,0.022100,16.281353,0.003186,0.003311,13.676228,0.000574,0.000594,24.0
1,-0.002647,0.019717,-1.241225,3.752492,0.019894,0.126251,17.398293,0.002852,0.003758,16.319873,0.000755,0.001207,24.0
2,-0.000309,0.016524,0.556773,5.411445,0.016527,0.122869,18.173261,0.019921,0.020828,17.198748,0.017672,0.018392,22.0
3,0.000412,0.005961,0.168945,-0.605352,0.005976,0.026902,16.635780,0.001028,0.001355,19.487258,0.000699,0.000739,20.0
4,-0.000665,0.007063,-0.133148,-0.799151,0.007094,0.028969,16.287375,0.002747,0.002827,18.095449,0.000590,0.000623,22.0


In [58]:
subjects = np.repeat(np.arange(53), 31)
subjects = subjects[:1638]  # trim extra if any

print("Subjects:", subjects.shape)

Subjects: (1638,)


In [59]:
print("Check lengths:", len(X), len(y), len(subjects))

Check lengths: 1638 1638 1638


In [60]:
# Feature columns (12 features)
feature_cols = [
    "mean",
    "std",
    "skew",
    "kurtosis",
    "rms",
    "ptp",
    "dominant_freq",
    "resp_power",
    "total_power",
    "rr_dominant_freq",
    "rr_resp_power",
    "rr_total_power"
]

X = df[feature_cols]   # Keep as DataFrame (important)
y = df["RR"]

In [62]:
gkf = GroupKFold(n_splits=5)

lgbm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LGBMRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        num_leaves=31,
        random_state=42,
        verbose=-1
    ))
])

mae_scores = []
rmse_scores = []
r2_scores = []

for fold, (train_idx, test_idx) in enumerate(gkf.split(X, y, groups=subjects)):
    
    X_train = X.iloc[train_idx]
    X_test  = X.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test  = y.iloc[test_idx]
    
    lgbm_pipeline.fit(X_train, y_train)
    y_pred = lgbm_pipeline.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    mae_scores.append(mae)
    rmse_scores.append(rmse)
    r2_scores.append(r2)
    
    print(f"Fold {fold+1}")
    print(f"MAE  : {mae:.3f} bpm")
    print(f"RMSE : {rmse:.3f} bpm")
    print(f"R²   : {r2:.3f}")
    print("-" * 30)

print("\n Cross-Validated Results")
print(f"Mean MAE  : {np.mean(mae_scores):.3f} ± {np.std(mae_scores):.3f}")
print(f"Mean RMSE : {np.mean(rmse_scores):.3f}")
print(f"Mean R²   : {np.mean(r2_scores):.3f}")

C:\Users\admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 1
MAE  : 2.375 bpm
RMSE : 3.253 bpm
R²   : -0.366
------------------------------


C:\Users\admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 2
MAE  : 2.287 bpm
RMSE : 3.083 bpm
R²   : 0.023
------------------------------


C:\Users\admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 3
MAE  : 2.150 bpm
RMSE : 3.037 bpm
R²   : -0.084
------------------------------


C:\Users\admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 4
MAE  : 2.680 bpm
RMSE : 4.107 bpm
R²   : -0.355
------------------------------
Fold 5
MAE  : 2.137 bpm
RMSE : 2.726 bpm
R²   : 0.096
------------------------------

 Cross-Validated Results
Mean MAE  : 2.326 ± 0.198
Mean RMSE : 3.241
Mean R²   : -0.137


C:\Users\admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [63]:
final_lgbm = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LGBMRegressor(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        num_leaves=31,
        random_state=42
    ))
])

final_lgbm.fit(X, y)

,steps,"[('scaler', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,boosting_type,'gbdt'
,num_leaves,31
,max_depth,6
,learning_rate,0.05


In [64]:
y_pred_full = final_lgbm.predict(X)

mae_full = mean_absolute_error(y, y_pred_full)
rmse_full = np.sqrt(mean_squared_error(y, y_pred_full))
r2_full = r2_score(y, y_pred_full)

print("\n Full Data Training Metrics")
print(f"MAE  : {mae_full:.3f} bpm")
print(f"RMSE : {rmse_full:.3f} bpm")
print(f"R²   : {r2_full:.3f}")


 Full Data Training Metrics
MAE  : 0.817 bpm
RMSE : 1.096 bpm
R²   : 0.879


C:\Users\admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [65]:
model_path = r"C:\Users\admin\OneDrive\Desktop\healthcare_ppg\models\LGBM_rr_pipeline.pkl"

joblib.dump(final_lgbm, model_path)

print("LGBM model saved successfully.")

LGBM model saved successfully.
